In [1]:
print("Đang cài đặt các thư viện hệ thống nâng cao...")
!pip install -q langchain langchain-community langchain-text-splitters pypdf chromadb sentence-transformers transformers accelerate bitsandbytes gradio langchain-huggingface langchain-chroma rank_bm25
print("-> Cài đặt hoàn tất!")

Đang cài đặt các thư viện hệ thống nâng cao...
-> Cài đặt hoàn tất!


In [2]:
import os
import shutil
import json
import numpy as np
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

INPUT_DIR = "/kaggle/input/datasets/phtranth/rag-asset"
WORKING_DB_PATH = "/kaggle/working/chromadb_hybrid_parent_child_v4"

print("1. Đang đồng bộ VectorDB sang thư mục làm việc (Working directory)...")
if os.path.exists(WORKING_DB_PATH):
    shutil.rmtree(WORKING_DB_PATH)

src_db_path = os.path.join(INPUT_DIR, "chromadb_hybrid_parent_child_v4")
shutil.copytree(src_db_path, WORKING_DB_PATH)
print("-> Đồng bộ thành công!")

print("\n2. Đang nạp lại kho dữ liệu đoạn Cha (parent_store)...")
parent_store_file = os.path.join(INPUT_DIR, "parent_store.json")
with open(parent_store_file, "r", encoding="utf-8") as f:
    parent_store = json.load(f)
print(f"-> Nạp thành công {len(parent_store)} đoạn Cha!")

print("\n3. Đang kết nối lại Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="keepitreal/vietnamese-sbert", model_kwargs={'device': 'cuda'})
vectordb = Chroma(persist_directory=WORKING_DB_PATH, embedding_function=embeddings)
vector_retriever = vectordb.as_retriever(search_kwargs={"k": 8})

print("\n4. Đang khôi phục bộ tìm kiếm từ khóa BM25...")
db_data = vectordb.get(include=["documents", "metadatas"])
child_docs = []
for text, meta in zip(db_data["documents"], db_data["metadatas"]):
    child_docs.append(Document(page_content=text, metadata=meta))

bm25_retriever = BM25Retriever.from_documents(child_docs)
bm25_retriever.k = 8

print("\n=== ĐÃ KHỞI TẠO XONG HẠ TẦNG TÌM KIẾM LAI CHA-CON TỪ INPUT DATASET! ===")

/tmp/ipykernel_285/2347532780.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_285/2347532780.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="keepitreal/vietnamese-sbert", model_kwargs={'device': 'cuda'})


1. Đang đồng bộ VectorDB sang thư mục làm việc (Working directory)...
-> Đồng bộ thành công!

2. Đang nạp lại kho dữ liệu đoạn Cha (parent_store)...
-> Nạp thành công 138 đoạn Cha!

3. Đang kết nối lại Vector Database...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



4. Đang khôi phục bộ tìm kiếm từ khóa BM25...

=== ĐÃ KHỞI TẠO XONG HẠ TẦNG TÌM KIẾM LAI CHA-CON TỪ INPUT DATASET! ===


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig, AutoModelForSequenceClassification
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"
print("1. Đang tải mô hình ngôn ngữ lớn Qwen-7B (Cấu hình lượng tử hóa 4-bit)...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=quant_config, 
    device_map={"": 0} 
)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=450, 
    do_sample=False,        
    repetition_penalty=1.0, 
    eos_token_id=terminators,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)
print("-> Tải Qwen-7B thành công!")

reranker_id = "BAAI/bge-reranker-base"
print(f"\n2. Đang tải mô hình Reranker: {reranker_id}...")
reranker_tokenizer = AutoTokenizer.pretrained_tokenizer = AutoTokenizer.from_pretrained(reranker_id)
reranker_model = AutoModelForSequenceClassification.from_pretrained(reranker_id).to("cuda:0")
print("-> Tải Reranker thành công!")

print("\n=== ĐÃ KHỞI TẠO XONG TOÀN BỘ BỘ NÃO AI TRÊN GPU 0! ===")

1. Đang tải mô hình ngôn ngữ lớn Qwen-7B (Cấu hình lượng tử hóa 4-bit)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'eos_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


-> Tải Qwen-7B thành công!

2. Đang tải mô hình Reranker: BAAI/bge-reranker-base...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Tải Reranker thành công!

=== ĐÃ KHỞI TẠO XONG TOÀN BỘ BỘ NÃO AI TRÊN GPU 0! ===


In [4]:
import json
import random
import gradio as gr
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def retrieve_hybrid_reranked(question, k_top=3):
    bm25_docs = bm25_retriever.invoke(question)
    vector_docs = vector_retriever.invoke(question)
    
    candidates = []
    seen_content = set()
    for doc in bm25_docs + vector_docs:
        if doc.page_content not in seen_content:
            seen_content.add(doc.page_content)
            candidates.append(doc)
            
    if not candidates:
        return []
        
    pairs = [[question, doc.page_content] for doc in candidates]
    with torch.no_grad():
        inputs = reranker_tokenizer(pairs, padding=True, truncation=True, return_tensors='pt').to("cuda:0")
        scores = reranker_model(**inputs).logits.view(-1).float().cpu().numpy()
        
    ranked_indices = np.argsort(scores)[::-1]
    ranked_docs = [candidates[idx] for idx in ranked_indices]
    return ranked_docs[:k_top]

def query_rag_general(question):
    child_docs = retrieve_hybrid_reranked(question, k_top=3)
    
    seen_parent_ids = set()
    parent_contexts = []
    
    for doc in child_docs:
        p_id = doc.metadata.get("parent_id")
        if p_id and p_id not in seen_parent_ids:
            seen_parent_ids.add(p_id)
            parent_contexts.append(parent_store[p_id]) 
            
    context = "\n\n".join(parent_contexts)
    
    messages = [
        {
            "role": "system", 
            "content": """Bạn là một trợ lý ảo chuyên nghiệp có nhiệm vụ trích xuất tài liệu pháp lý và kỹ thuật.

QUY TẮC BẮT BUỘC:
1. Hãy tìm chính xác phân đoạn hoặc đề mục trong "Tài liệu" trực tiếp giải quyết câu hỏi của người dùng.
2. Trích xuất Y NGUYÊN VĂN đoạn quy định đó ra màn hình. Tuyệt đối không được thay đổi từ ngữ, không tự ý tóm tắt, không viết lại (paraphrase).
3. BẮT BUỘC ghi rõ tiêu đề nguồn ở đầu câu trả lời dựa trên nhãn "[Trích xuất từ <Tiêu đề> của tài liệu]" có sẵn ở đầu mỗi đoạn đọc. 
   Định dạng đầu ra bắt buộc viết hoa tiêu đề nguồn như sau:
   "Theo thông tin trích xuất từ <Tiêu đề> của tài liệu:
   - [Đoạn văn sao chép nguyên văn]"
4. CẤM viết lời chào hỏi, không viết lời dẫn dắt khác, không ký tên ở cuối. Chỉ in ra đúng cấu trúc yêu cầu.
5. Nếu tài liệu hoàn toàn không chứa câu trả lời trực tiếp cho câu hỏi, chỉ trả lời duy nhất: "Không tìm thấy thông tin phù hợp trong tài liệu" và không giải thích gì thêm."""
        },
        {
            "role": "user", 
            "content": f"Tài liệu:\n{context}\n\nCâu hỏi: {question}"
        }
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = pipe(prompt)
    response = output[0]['generated_text'].strip()
    return response.split("<|im_start|>assistant")[-1].strip()

try:
    golden_file_path = "/kaggle/input/datasets/phtranth/rag-asset/golden_dataset.json"
    with open(golden_file_path, "r", encoding="utf-8") as f:
        golden_data = json.load(f)
    example_questions = [item["question"] for item in random.sample(golden_data, min(3, len(golden_data)))]
except Exception as e:
    example_questions = ["Đặt câu hỏi tra cứu của bạn tại đây!"]

def gradio_chat_fn(message, history):
    return query_rag_general(message)

demo = gr.ChatInterface(
    fn=gradio_chat_fn,
    title="🏐 Trợ lý AI Tra Cứu Vạn Năng - Enterprise Edition",
    description="Hệ thống RAG trích xuất thông tin dựa trên tài liệu PDF của bạn.",
    examples=example_questions,
    theme=gr.themes.Soft()
)

print("Đang khởi động Giao diện Web...")
demo.launch(share=True, debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Đang khởi động Giao diện Web...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://40b87f739261974140.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://40b87f739261974140.gradio.live
